# Documentação do Notebook

## Introdução

Este notebook foi desenvolvido para gerar gráficos a partir de várias simulações do muar.py . Ele está organizado para facilitar a execução e a compreensão dos resultados obtidos. Esta documentação fornecerá todas as informações necessárias para que você possa rodar o notebook corretamente e entender sua estrutura.

## Autor

Rodrigo Flexa  
Email: rodrigoflexa0211@gmail.com

## Requisitos

Para executar este notebook, você precisará ter instalados os seguintes pacotes:

- Python 3.x
- Jupyter Notebook
- Matplotlib
- Pandas
- Plotly
- Glob
- Typing


### Instalação via pip

Para instalar os pacotes necessários utilizando `pip`, você pode usar os seguintes comandos:

```bash
pip install matplotlib pandas plotly glob2

```
### Instalação via anaconda

Para instalar os pacotes necessários utilizando `conda`, você deve ter algum amabiente com o anaconda já instalado com o pyrthon 3.x ou então criar um do zero com os seguinte comando:

```bash
conda create -n meu_ambiente python=3.x
conda activate meu_ambiente
conda install matplotlib pandas plotly
pip install glob2
```


## Conteúdo

Esse notebook propõe uma análise dos resultados de métricas voltadas para as ferramentas 
de bitrate adaptativo, latência temporária, resiliência e também métricas usuais da simulação,
tais como consumo de cpu, cache, banda, reúso e etc, compondo portanto os seguintes gráficos:


- [Gráfico 1](#gr1): Eixo x porcentagem de confiabilidade, Eixo y métricas usuais da simulação inteira (Não ao longo do tempo)

- [Gráfico 2](#gr2): Eixo x Número de servidores derrubados. Eixo y métricas usuais

- [Gráfico 3](#gr3): Bit rate médio por sessão. Bit rate médio por usuário. Bit rate de usuários com conteúdo adaptado

- [Gráfico 4](#gr4): Avaliar usuários bem servidos. Colocar confiabilidade. 
Eixo y usuários bem e mal servidos. Eixo x um com % de confiabilidade e outro com quantidade de servidores derrubados


## Observações

- Esse notebook trabalha com um número fixo de usuários e também de sessões, variando apenas a confiabilidade dos servidores.

### Configuração inicial

Importando bibliotecas necessárias

In [2]:
import glob
import os
import pandas as pd

# import seaborn as sns
# import plotly.graph_objects as go
# import plotly.io as pio
# import plotly.express as px


import warnings

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

### Carregamento dos dados

In [3]:
results_flows_directories = glob.glob(
    "../../results/results_flows*/*"
)  # Caso mude a estrutura dos diretórios, esse caminho deve ser alterado
results_flows_directories

['../../results/results_flows/msf_s_50_p_4_sfc_on_rel_0.95',
 '../../results/results_flows/musfico_s_50_p_4_sfc_on_rel_0.975',
 '../../results/results_flows/ga_s_50_p_4_sfc_on_rel_0.99',
 '../../results/results_flows/osfem_s_50_p_4_sfc_on_rel_0.975',
 '../../results/results_flows/ga_s_50_p_4_sfc_on_rel_0.95',
 '../../results/results_flows/msf_s_50_p_4_sfc_on_rel_0.975',
 '../../results/results_flows/musfico_s_50_p_4_sfc_on_rel_0.99',
 '../../results/results_flows/osfem_s_50_p_4_sfc_on_rel_0.95',
 '../../results/results_flows/ga_s_50_p_4_sfc_on_rel_0.975',
 '../../results/results_flows/osfem_s_50_p_4_sfc_on_rel_0.99',
 '../../results/results_flows/musfico_s_50_p_4_sfc_on_rel_0.95',
 '../../results/results_flows/msf_s_50_p_4_sfc_on_rel_0.99']

### Função de coleta dos dados

Essa função irá basicamente métricas selecioandas e concatenar os csv's da simulação verticalmente, recortando as simulações no tempo 0 até 999.

Como os dados estarão concatenados, precisaremos posteriormente retirar uma média dessas simulações

In [25]:
aggregated_simulation_data = pd.DataFrame()


def collect_data_from_alg_directory(results_flows_directories, metricas_de_coleta):
    data_nla = []
    log_simu = {}  # Essa variável serve apenas para verificar as simulações

    for alg_dir in results_flows_directories:
        simulacoes_okays = 0
        simu_exec_name = alg_dir.split("/")[-1].split("_")
        # print(simu_exec_name)
        alg_name = simu_exec_name[0]
        reliability = simu_exec_name[-1]

        files = os.listdir(alg_dir)
        # print(f"Simulação: {alg_name}, {reliability}")
        # print("Quantidade de csv: ", len(files))

        if alg_name not in log_simu:
            log_simu[alg_name] = {
                "reliability": [reliability],
                "files": len(files),
                "simulacoes_sucesso": 0,
                "Nulos": 0,
            }
        else:
            log_simu[alg_name]["reliability"].append(reliability)
            log_simu[alg_name]["files"] += len(files)

        for file in files:
            data_path = os.path.join(alg_dir, file)
            try:
                simulation_df = pd.read_csv(data_path)
                print(simulation_df)
            except Exception as e:
                print(f"Erro ao ler o arquivo {file}: {e}")
                continue

            primeiro_tempo = simulation_df["timestamp"].values[0]
            simulation_df["tempo"] = simulation_df[["timestamp"]].applymap(
                lambda x: x - primeiro_tempo
            )
            simulation_is_success = "sfc_cache_p4_50" in simulation_df["sfc_id"].values

            if simulation_is_success:
                simulacoes_okays += 1
                simulation_df = simulation_df[metricas_de_coleta]

                # Arredondar tempo
                simulation_df["tempo"] = simulation_df["tempo"].astype(int)
                simulation_df = simulation_df.replace("None", pd.NA)

                latency_col = simulation_df[["tempo", "latency"]]
                latency_col.dropna(inplace=True)
                latency_col.loc[:, "latency"] = latency_col["latency"].astype(float)

                # Calcular média acumulada de 'success'
                cumulative_sum_success = 0
                cumulative_avg_success = []
                for i, value in enumerate(simulation_df["success"]):
                    cumulative_sum_success += value
                    cumulative_avg_success.append(cumulative_sum_success / (i + 1))
                simulation_df["success"] = cumulative_avg_success

                simulation_df = simulation_df.groupby("tempo", as_index=False).mean(
                    numeric_only=True
                )
                latency_df = (
                    latency_col.groupby("tempo", as_index=False)
                    .mean(numeric_only=True)
                    .reset_index()
                )

                tempo_range = simulation_df["tempo"].max()
                df_mean = simulation_df.set_index("tempo").reindex(range(tempo_range + 1))
                df_mean = df_mean.fillna(method="ffill")
                df_mean = df_mean.reset_index()

                latency_df = latency_df.set_index("tempo").reindex(range(tempo_range + 1))
                latency_df = latency_df.fillna(method="ffill")
                latency_df = latency_df.reset_index()

                df_mean["latency"] = latency_df["latency"]
                df_mean["algorithm"] = alg_name
                df_mean["reliability"] = reliability
                df_mean = df_mean.iloc[0:1000]
                data_nla.append(df_mean)

        log_simu[alg_name]["simulacoes_sucesso"] += simulacoes_okays
        log_simu[alg_name]["Nulos"] += sum(simulation_df.isnull().sum())

    if data_nla:
        data_nla_f = pd.concat(data_nla)
    else:
        data_nla_f = pd.DataFrame()

    print("\nResumo das Simulações:")
    for alg, info in log_simu.items():
        print(f"Algoritmo: {alg}")
        print(f"  Confiabilidades: {info['reliability']}")
        print(f"  Quantidade de arquivos: {info['files']}")
        print(f"  Simulações de sucesso: {info['simulacoes_sucesso']}")
        print(f"  Dados nulos: {info['Nulos']}")
        print()

    return data_nla_f

### Uso da função de coleta dos dados

Selecione quais métricas (colunas) serão coletadas com base no csv flows.

In [26]:
metricas_de_coleta = [
    "tempo",
    "cpu_utilization",
    "cache_utilization",
    "bandwidth_utilization",
    "cpu_resilient",
    "cache_resilient",
    "bw_resilient",
    "success",
    "latency",
    "duration",
    "running_sfcs",
    "running_players",
    "running_sessions",
    "cpu_saved",
    "cache_saved",
    "shared_vnfs",
    "server_crashed",
]

In [27]:
aggregated_simulation_data = collect_data_from_alg_directory(
    results_flows_directories, metricas_de_coleta
)

      No.          time  timestamp  number_of_sfc  cpu_utilization  \
1     NaN  1.718900e+09          4         0.0103           0.0617   
2     NaN  1.718900e+09          4         0.0317           0.1900   
3     NaN  1.718900e+09          4         0.0333           0.2000   
4     NaN  1.718900e+09          4         0.0547           0.3283   
5     NaN  1.718900e+09          4         0.0564           0.3383   
...   ...           ...        ...            ...              ...   
1321  NaN  1.718901e+09          4         0.7029           0.7029   
1322  NaN  1.718901e+09          4         0.7054           0.7054   
1323  NaN  1.718901e+09          4         0.7375           0.7375   
1324  NaN  1.718901e+09          4         0.7400           0.7400   
1325  NaN  1.718901e+09          4         0.7400           0.7400   

      cpu_active_servers  bandwidth_utilization  bw_active_links  \
1                 0.0156                 0.1362           0.0333   
2                 0.046

      No.          time  timestamp  number_of_sfc  cpu_utilization  \
1     NaN  1.718900e+09          4         0.0103           0.1233   
2     NaN  1.718900e+09          4         0.0317           0.3800   
3     NaN  1.718900e+09          4         0.0333           0.4000   
4     NaN  1.718900e+09          4         0.0547           0.6567   
5     NaN  1.718900e+09          4         0.0564           0.6767   
...   ...           ...        ...            ...              ...   
1653  NaN  1.718901e+09          4         0.5852           0.7152   
1654  NaN  1.718901e+09          4         0.5964           0.6560   
1655  NaN  1.718901e+09          4         0.6197           0.6817   
1656  NaN  1.718901e+09          4         0.6215           0.6837   
1657  NaN  1.718901e+09          4         0.6448           0.7093   

      cpu_active_servers  bandwidth_utilization  bw_active_links  \
1                 0.0018                 0.0647           0.0333   
2                 0.005

      No.          time  timestamp  number_of_sfc  cpu_utilization  \
1     NaN  1.718900e+09          4         0.0103           0.1233   
2     NaN  1.718900e+09          4         0.0317           0.3800   
3     NaN  1.718900e+09          4         0.0333           0.4000   
4     NaN  1.718900e+09          4         0.0547           0.6567   
5     NaN  1.718900e+09          4         0.0564           0.6767   
...   ...           ...        ...            ...              ...   
1488  NaN  1.718901e+09          4         0.5437           0.6117   
1489  NaN  1.718901e+09          4         0.5574           0.5574   
1490  NaN  1.718901e+09          4         0.5859           0.5859   
1491  NaN  1.718901e+09          4         0.5881           0.5881   
1492  NaN  1.718901e+09          4         0.6167           0.6167   

      cpu_active_servers  bandwidth_utilization  bw_active_links  \
1                 0.0018                 0.0647           0.0333   
2                 0.005

      No.          time  timestamp  number_of_sfc  cpu_utilization  \
1     NaN  1.718900e+09          4         0.0103           0.0617   
2     NaN  1.718900e+09          4         0.0317           0.1900   
3     NaN  1.718900e+09          4         0.0333           0.2000   
4     NaN  1.718900e+09          4         0.0547           0.3283   
5     NaN  1.718900e+09          4         0.0564           0.3383   
...   ...           ...        ...            ...              ...   
1377  NaN  1.718901e+09          4         0.4097           0.4097   
1378  NaN  1.718901e+09          4         0.4117           0.4117   
1379  NaN  1.718901e+09          4         0.4373           0.4373   
1380  NaN  1.718901e+09          4         0.4393           0.4393   
1381  NaN  1.718901e+09          4         0.4650           0.4650   

      cpu_active_servers  bandwidth_utilization  bw_active_links  \
1                 0.0156                 0.1362           0.0333   
2                 0.046

In [24]:
print(aggregated_simulation_data)

Empty DataFrame
Columns: []
Index: []


### Criação de colunas extras

Aqui devem ser criadas as métricas novas a partir de outras, como por exemplo CPU consumida por Fluxo

In [20]:
aggregated_simulation_data["CPU Salva por Fluxo"] = (
    aggregated_simulation_data["cpu_saved"] / aggregated_simulation_data["running_sfcs"]
)
aggregated_simulation_data["CPU Salva por Servidor"] = aggregated_simulation_data["cpu_saved"] / 35

aggregated_simulation_data["Cache Salva por Fluxo"] = (
    aggregated_simulation_data["cache_saved"] / aggregated_simulation_data["running_sfcs"]
)
aggregated_simulation_data["Cache Salva por Servidor"] = (
    aggregated_simulation_data["cache_saved"] / 35
)

aggregated_simulation_data["CPU por Fluxo"] = (
    aggregated_simulation_data["cpu_utilization"] / aggregated_simulation_data["running_sfcs"]
)
aggregated_simulation_data["Cache por Fluxo"] = (
    aggregated_simulation_data["cache_utilization"] / aggregated_simulation_data["running_sfcs"]
)
aggregated_simulation_data["Banda por Fluxo"] = (
    aggregated_simulation_data["bandwidth_utilization"]
    / aggregated_simulation_data["running_sfcs"]
)

# Usuários mal servidos: 0
# Usuários bem servidos: 1
aggregated_simulation_data["servido"] = aggregated_simulation_data["latency"].apply(
    lambda x: 1 if 0 <= x <= 6 else 0
)

KeyError: 'cpu_saved'

### Fase de agrupamento dos dados

As simulações estão concatenadas verticalmente, precisamo tirar agora uma média geral dessas simulações, agrupando por algoritmo e também pela reliability

In [7]:
# Agrupar os dados por algoritmo e confiabilidade
grouped = aggregated_simulation_data.groupby(["algorithm", "reliability", "tempo"]).mean()
grouped

cpu_utilization  cache_utilization  \
algorithm reliability tempo                                       
ga        0.95        0             0.030028           0.031900   
                      1             0.044868           0.040425   
                      2             0.044006           0.046475   
                      3             0.078269           0.075350   
                      4             0.075785           0.062883   
...                                      ...                ...   
osfem     0.99        995           0.626000           0.415700   
                      996           0.622945           0.411200   
                      997           0.624820           0.411200   
                      998           0.633295           0.414450   
                      999           0.624570           0.411575   

                             bandwidth_utilization  cpu_resilient  \
algorithm reliability tempo                                         
ga        0.95        0                   0.012124       0.030028   
                      1                   0.011995       0.044868   
                      2                   0.013855       0.044006   
                      3                   0.027585       0.078269   
                      4                   0.028787       0.075785   
...                                            ...            ...   
osfem     0.99        995                 0.102090       0.562745   
                      996                 0.116182       0.558165   
                      997                 0.103113       0.561830   
                      998                 0.110607       0.569045   
                      999                 0.105353       0.562285   

                             cache_resilient  bw_resilient   success  \
algorithm reliability tempo                                            
ga        0.95        0             0.031900      0.012124  1.000000   
                      1             0.040425      0.011995  1.000000   
                      2             0.046475      0.013855  1.000000   
                      3             0.075350      0.027585  1.000000   
                      4             0.062883      0.028787  1.000000   
...                                      ...           ...       ...   
osfem     0.99        995           0.374000      0.112160  0.997374   
                      996           0.367125      0.101452  0.997376   
                      997           0.369875      0.102283  0.997377   
                      998           0.371250      0.115998  0.997380   
                      999           0.369875      0.105353  0.997380   

                              latency  duration  running_sfcs  ...  \
algorithm reliability tempo                                    ...   
ga        0.95        0      2.200000  0.177219          2.84  ...   
                      1      1.490000  0.188888          3.38  ...   
                      2      1.850000  0.181434          3.40  ...   
                      3      1.650000  0.173734          6.24  ...   
                      4      1.856667  0.178457          5.82  ...   
...                               ...       ...           ...  ...   
osfem     0.99        995    1.175000  0.000699         49.80  ...   
                      996    1.250000  0.000499         49.65  ...   
                      997    1.450000  0.000647         49.90  ...   
                      998    1.350000  0.000698         50.25  ...   
                      999    1.100000  0.000598         49.65  ...   

                             shared_vnfs  server_crashed  CPU Salva por Fluxo  \
algorithm reliability tempo                                                     
ga        0.95        0             2.04             0.0             1.636364   
                      1             0.58             0.0             0.138889   
                      2             0.78             0.0             0.000000   
      

In [8]:
# Caso queira acessa um algoritmo com alguma reliability específica:
# filtered_group = final_grouped_data.xs(('osfem', 0.95), level=['algorithm', 'reliability'])
# filtered_group

# Gráfico 1 <a id="gr1"></a>

Nesse caso queremos uma média da simulação inteira e não uma análise ao longo do tempo, portanto usaremos o seguinte agrupamento:

In [9]:
grouped_df = aggregated_simulation_data.groupby(["algorithm", "reliability"]).mean().reset_index()
grouped_df

,algorithm,reliability,tempo,cpu_utilization,cache_utilization,bandwidth_utilization,cpu_resilient,cache_resilient,bw_resilient,success,...,shared_vnfs,server_crashed,CPU Salva por Fluxo,CPU Salva por Servidor,Cache Salva por Fluxo,Cache Salva por Servidor,CPU por Fluxo,Cache por Fluxo,Banda por Fluxo,servido
0,ga,0.95,499.5,0.585837,0.533550,0.143195,0.374399,0.342435,0.143195,0.923652,...,18.440797,0.003167,0.880647,0.794472,3.188801,2.885121,0.019685,0.017662,0.004704,0.993800
1,ga,0.975,499.5,0.523300,0.510725,0.169091,0.443761,0.432929,0.169091,0.950982,...,21.558693,0.001833,0.887236,0.937440,3.169440,3.318009,0.014120,0.013710,0.004561,0.995800
2,ga,0.99,499.5,0.516318,0.486030,0.172828,0.472658,0.445061,0.172828,0.966034,...,24.842623,0.000867,0.990280,1.133063,3.521913,4.042085,0.013004,0.012272,0.004338,0.999600
3,msf,0.95,499.5,0.662089,0.538459,0.226507,0.478258,0.388329,0.226507,0.941790,...,26.937646,0.002133,1.107707,1.168468,4.175248,4.372029,0.016905,0.013598,0.005708,0.972500
4,msf,0.975,499.5,0.531333,0.420925,0.195187,0.458137,0.361514,0.195187,0.971211,...,36.578507,0.004100,1.530060,1.686784,5.778183,6.352767,0.013440,0.010545,0.004876,0.991000
5,msf,0.99,499.5,0.548558,0.447707,0.185071,0.467975,0.380633,0.185071,0.973901,...,35.641967,0.001100,1.466842,1.632130,5.502115,6.104808,0.013603,0.010987,0.004542,0.987800
6,musfico,0.95,499.5,0.722899,0.586758,0.202181,0.495309,0.395093,0.202181,0.929877,...,42.348807,0.002500,1.507341,1.824787,5.913167,7.149953,0.016883,0.013514,0.004690,0.964000
7,musfico,0.975,499.5,0.573477,0.438358,0.179569,0.472247,0.359142,0.179569,0.976716,...,43.789877,0.002867,1.675290,1.938261,6.470542,7.495708,0.013939,0.010514,0.004288,0.972400
8,musfico,0.99,499.5,0.736586,0.631408,0.236723,0.612321,0.519366,0.236723,0.909143,...,47.962767,0.001800,1.417157,1.998566,5.504775,7.761279,0.013840,0.011548,0.004437,0.972200
9,osfem,0.95,499.5,0.672165,0.470052,0.118375,0.475100,0.326893,0.118462,0.993012,...,45.254126,0.001735,1.865263,2.230717,7.402184,8.855118,0.015931,0.011165,0.002764,0.998625


In [16]:
def plot_metric_1(grouped_df, metric, algorithms=None):
    """
    Função para plotar um gráfico de barras para uma métrica específica e algoritmos selecionados.

    Parâmetros:
    - grouped_df: DataFrame agrupado com as métricas.
    - metric: Métrica a ser plotada (e.g., 'cpu_utilization', 'cache_utilization').
    - algorithms: Lista de algoritmos a serem mostrados (opcional). Se None, mostra todos os algoritmos.
    """
    if algorithms:
        filtered_df = grouped_df[grouped_df["algorithm"].isin(algorithms)]
    else:
        filtered_df = grouped_df

    fig = px.bar(
        filtered_df,
        x="reliability",
        y=metric,
        color="algorithm",
        barmode="group",
        title=f"{metric.replace('_', ' ').title()} vs Reliability",
    )

    fig.update_layout(xaxis_title="Reliability", yaxis_title=metric.replace("_", " ").title())

    fig.show()

In [17]:
# Exemplo de uso da função
plot_metric_1(grouped_df, "success", algorithms=["musfico", "msf", "ga", "osfem"])
plot_metric_1(grouped_df, "cpu_resilient", algorithms=["musfico", "msf", "ga", "osfem"])
plot_metric_1(grouped_df, "cache_resilient", algorithms=["musfico", "msf", "ga", "osfem"])
plot_metric_1(grouped_df, "bw_resilient", algorithms=["musfico", "msf", "ga", "osfem"])

# Gráfico 2 <a id="gr2"></a>

In [11]:
# TODO ainda não desenvolvido

# Gráfico 3 <a id="gr3"></a>

In [12]:
# TODO ainda não desenvolvido

# Gráfico 4 <a id="gr4"></a>

In [13]:
# TODO ainda não desenvolvido